B6. Конформность: предсказать порчу ответа по данным R1 (Hao 2026, Sharma 2023).....


In [1]:
import sys, os, json

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/panel_mechanisms')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
import statsmodels.api as sm
PANEL_REL = {'P-base': PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv', 'P-weak': PANELS_OTHER_DIR['P-weak'] + '/llm_as_judge/' + PANEL_FILE['P-weak'], 'P-mixed': PANELS_OTHER_DIR['P-mixed'] + '/llm_as_judge/' + PANEL_FILE['P-mixed'], 'P-strong': PANELS_OTHER_DIR['P-strong'] + '/llm_as_judge/' + PANEL_FILE['P-strong'], 'homog gpt': HOMOG_DIRS['gpt'] + '/llm_as_judge/delphi_homog_gpt_noroles_judge.csv', 'homog qwen': HOMOG_DIRS['qwen'] + '/llm_as_judge/delphi_homog_qwen_noroles_judge.csv', 'homog gemini': HOMOG_DIRS['gemini'] + '/llm_as_judge/delphi_homog_gemini_noroles_judge.csv', 'homog deepseekv4': HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'}


In [2]:
def build_long(arm, rel):
    d = rd(rel)
    if d is None or 'Type' not in d.columns:
        return None
    pids = participants(d)
    if not pids:
        return None
    truth = true_label(d)
    lab = {}
    for r in (1, 2, 3):
        for p in pids:
            c = 'r%d_emergency__%s' % (r, p)
            lab[r, p] = d[c].map(norm_label) if c in d.columns else pd.Series(index=d.index, dtype=object)
    R1 = pd.DataFrame({p: lab[1, p] for p in pids})
    n_voters = R1.notna().sum(axis=1)

    def ent(row):
        v = row.dropna()
        if not len(v):
            return np.nan
        pr = v.value_counts(normalize=True).to_numpy()
        return float(-(pr * np.log(pr)).sum())
    entropy = R1.apply(ent, axis=1)
    modal = R1.mode(axis=1)
    modal = modal[0] if len(modal.columns) else pd.Series(index=d.index, dtype=object)
    rows = []
    for p in pids:
        l1 = lab[1, p]
        last = lab[3, p].where(lab[3, p].notna(), lab[2, p])
        last = last.where(last.notna(), l1)
        conf = pd.to_numeric(d.get('r1_confidence__%s' % p), errors='coerce')
        ntok = pd.to_numeric(d.get('r1_completion_tokens__%s' % p), errors='coerce')
        nsub = pd.to_numeric(d.get('r1_n_subquestions__%s' % p), errors='coerce')
        agree = R1.drop(columns=[p]).eq(l1, axis=0).sum(axis=1)
        others = R1.drop(columns=[p]).notna().sum(axis=1)
        rows.append(pd.DataFrame({'arm': arm, 'case': d.index, 'pid': p, 'r1': l1.to_numpy(), 'last': last.to_numpy(), 'truth': truth.to_numpy(), 'conf': conf.to_numpy() if conf is not None else np.nan, 'r1_tokens': ntok.to_numpy() if ntok is not None else np.nan, 'n_subq': nsub.to_numpy() if nsub is not None else np.nan, 'allies': agree.to_numpy(), 'others': others.to_numpy(), 'entropy': entropy.to_numpy(), 'modal': modal.to_numpy(), 'n_voters': n_voters.to_numpy(), 'rounds_executed': pd.to_numeric(d.get('delphi_rounds_executed'), errors='coerce').to_numpy(), 'Type': d['Type'].to_numpy(), 'spec': d['Sub Clinical Specialties'].to_numpy()}))
    return pd.concat(rows, ignore_index=True)
L = pd.concat([x for x in (build_long(a, r) for a, r in PANEL_REL.items()) if x is not None], ignore_index=True)
L = L[L['r1'].notna() & L['truth'].notna()].copy()
L['r1_ok'] = (L['r1'] == L['truth']).astype(int)
L['last_ok'] = (L['last'] == L['truth']).astype(int)
L['spoke_again'] = (L['last'] != L['r1']) | (L['rounds_executed'] > 1)
L['changed'] = (L['last'] != L['r1']).astype(int)
L['minority'] = (L['r1'] != L['modal']).astype(int)
L['against'] = L['others'] - L['allies']
print('наблюдений (кейс × участник):', len(L))
print('плеч:', L['arm'].nunique(), ' участников всего:', L['pid'].nunique())


наблюдений (кейс × участник): 5415
плеч: 8  участников всего: 18


In [3]:
E = L[L['rounds_executed'] > 1].copy()
E['порча'] = ((E['r1_ok'] == 1) & (E['last_ok'] == 0)).astype(int)
E['починка'] = ((E['r1_ok'] == 0) & (E['last_ok'] == 1)).astype(int)
rows = []
for arm, g in E.groupby('arm', observed=True):
    n = len(g)
    sp, rp = (int(g['порча'].sum()), int(g['починка'].sum()))
    acc1, acc2 = (float(g['r1_ok'].mean()), float(g['last_ok'].mean()))
    tab = stats.binomtest(rp, sp + rp, 0.5).pvalue if sp + rp else np.nan
    rows.append({'плечо': arm, 'n набл.': n, 'сменили метку': int(g['changed'].sum()), 'порча': sp, 'починка': rp, 'нетто': rp - sp, 'точность R1': acc1, 'точность финал': acc2, 'Δ точности': acc2 - acc1, 'p (порча=починка)': float(tab) if tab == tab else np.nan})
S = pd.DataFrame(rows).set_index('плечо')
show(S, 'H6.1 Порча против починки на уровне участников')
tot_s, tot_r = (int(E['порча'].sum()), int(E['починка'].sum()))
print()
print('ИТОГО по всем панелям: порча %d, починка %d, нетто %+d' % (tot_s, tot_r, tot_r - tot_s))
print('точный биномиальный тест H0 «порча = починка»: p = %s' % fmt_p(stats.binomtest(tot_r, tot_s + tot_r, 0.5).pvalue if tot_s + tot_r else np.nan))



H6.1 Порча против починки на уровне участников
------------------------------------------------------------
                  n набл.  сменили метку  порча  починка  нетто  точность R1  точность финал  Δ точности  p (порча=починка)
плечо                                                                                                                      
P-base                327             66     26       39     13       0.6942          0.7339      0.0398             0.1360
P-mixed               261             44     12       32     20       0.6973          0.7739      0.0766             0.0037
P-strong              303             62     17       45     28       0.6865          0.7789      0.0924             0.0005
P-weak                326             48     15       32     17       0.7209          0.7730      0.0521             0.0186
homog deepseekv4      174             27     15       12     -3       0.7069          0.6897     -0.0172             0.7011
homog gemini          1

In [4]:
sub = E[E['r1_ok'] == 1]
ct = pd.crosstab(sub['minority'], sub['порча'], normalize='index')
cnt = pd.crosstab(sub['minority'], sub['порча'])
tab = pd.DataFrame({'n': cnt.sum(axis=1), 'испорчено': cnt.get(1, 0), 'доля порчи': ct.get(1, 0)})
tab.index = ['в большинстве R1', 'в меньшинстве R1']
show(tab, 'Доля порчи среди изначально ПРАВЫХ участников')
if cnt.shape == (2, 2):
    orat = cnt.iloc[1, 1] * cnt.iloc[0, 0] / max(cnt.iloc[1, 0] * cnt.iloc[0, 1], 1e-09)
    print('\nотношение шансов (меньшинство/большинство): %.2f' % orat)
    print('точный тест Фишера: p = %s' % fmt_p(stats.fisher_exact(cnt.to_numpy())[1]))



Доля порчи среди изначально ПРАВЫХ участников
------------------------------------------------------------
                     n  испорчено  доля порчи
в большинстве R1  1104         43      0.0389
в меньшинстве R1   104         68      0.6538

отношение шансов (меньшинство/большинство): 46.61
точный тест Фишера: p = 1.5e-54


In [5]:
sub2 = E[E['r1_ok'] == 0]
cnt2 = pd.crosstab(sub2['minority'], sub2['починка'])
tab2 = pd.DataFrame({'n': cnt2.sum(axis=1), 'починено': cnt2.get(1, 0), 'доля починки': pd.crosstab(sub2['minority'], sub2['починка'], normalize='index').get(1, 0)})
tab2.index = ['в большинстве R1', 'в меньшинстве R1']
show(tab2, 'Доля починки среди изначально НЕПРАВЫХ участников')



Доля починки среди изначально НЕПРАВЫХ участников
------------------------------------------------------------
                    n  починено  доля починки
в большинстве R1  380        72        0.1895
в меньшинстве R1  157       122        0.7771


In [6]:
F = E[E['r1_ok'] == 1].copy()
F['conf_z'] = (F['conf'] - F['conf'].mean()) / F['conf'].std(ddof=0)
F['tok_z'] = (np.log1p(F['r1_tokens']) - np.log1p(F['r1_tokens']).mean()) / np.log1p(F['r1_tokens']).std(ddof=0)
F['ent_z'] = (F['entropy'] - F['entropy'].mean()) / F['entropy'].std(ddof=0)
F['against_z'] = (F['against'] - F['against'].mean()) / F['against'].std(ddof=0)
F['hetero'] = F['arm'].str.startswith('P-').astype(int)
PRED = ['minority', 'against_z', 'ent_z', 'conf_z', 'tok_z', 'hetero']
F = F.dropna(subset=PRED + ['порча'])
X = sm.add_constant(F[PRED].astype(float))
y = F['порча'].astype(float)
m = sm.Logit(y, X).fit(disp=0, cov_type='cluster', cov_kwds={'groups': F['case'].astype('category').cat.codes})
res = pd.DataFrame({'коэф': m.params, 'SE': m.bse, 'z': m.tvalues, 'p': m.pvalues, 'OR': np.exp(m.params)})
show(res, 'Логистическая модель порчи (SE кластеризованы по кейсу)')
print('\nпсевдо-R2 (Макфадден): %.4f     n = %d,  событий = %d' % (m.prsquared, len(F), int(y.sum())))
print()
print('ВНИМАНИЕ на коллинеарность: `minority`, `against_z` и `ent_z` описывают')
print('одно и то же — расхождение участника с большинством. В полной модели они')
print('делят между собой один сигнал, поэтому по отдельности незначимы, хотя')
print('совместно объясняют почти всё (псевдо-R2 = %.2f).' % m.prsquared)
print('Корреляции между ними:')
show(F[['minority', 'against_z', 'ent_z', 'conf_z']].corr(), None)



Логистическая модель порчи (SE кластеризованы по кейсу)
------------------------------------------------------------
             коэф     SE       z      p     OR
const     -4.0291 0.6795 -5.9295 0.0000 0.0178
minority   1.1094 1.3684  0.8107 0.4175 3.0325
against_z  0.8548 0.8756  0.9763 0.3289 2.3510
ent_z      1.3806 0.7726  1.7870 0.0739 3.9774
conf_z     0.0005 0.1466  0.0037 0.9970 1.0005
tok_z     -0.0430 0.1877 -0.2291 0.8188 0.9579
hetero    -0.6740 0.3761 -1.7920 0.0731 0.5097

псевдо-R2 (Макфадден): 0.4633     n = 1208,  событий = 111

ВНИМАНИЕ на коллинеарность: `minority`, `against_z` и `ent_z` описывают
одно и то же — расхождение участника с большинством. В полной модели они
делят между собой один сигнал, поэтому по отдельности незначимы, хотя
совместно объясняют почти всё (псевдо-R2 = 0.46).
Корреляции между ними:


           minority  against_z   ent_z  conf_z
minority     1.0000     0.7425  0.4274 -0.1671
against_z    0.7425     1.0000  0.9198 -0.3087
ent_z        0.4274     0.9198  1.0000 -0.3183
conf_z      -0.1671    -0.3087 -0.3183  1.0000


In [7]:
for spec in (['minority'], ['minority', 'hetero'], ['against_z'], ['ent_z']):
    Xs = sm.add_constant(F[spec].astype(float))
    ms = sm.Logit(y, Xs).fit(disp=0, cov_type='cluster', cov_kwds={'groups': F['case'].astype('category').cat.codes})
    print('%-24s  ' % '+'.join(spec), end='')
    for k in spec:
        print('%s: OR=%6.2f p=%s   ' % (k, np.exp(ms.params[k]), fmt_p(ms.pvalues[k])), end='')
    print('  псевдо-R2=%.3f' % ms.prsquared)


minority                  minority: OR= 46.61 p=7.3e-35     псевдо-R2=0.329
minority+hetero           minority: OR= 46.09 p=3.7e-34   hetero: OR=  0.77 p=0.399   

  псевдо-R2=0.330
against_z                 against_z: OR=  6.37 p=6.1e-42     псевдо-R2=0.447
ent_z                     ent_z: OR= 10.57 p=9.9e-07     псевдо-R2=0.321


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
Xn, yn, grp = (F[PRED].to_numpy(float), y.to_numpy(), F['arm'].to_numpy())
rows = []
for label, mk in [('логистическая', lambda: LogisticRegression(max_iter=2000)), ('градиентный бустинг', lambda: GradientBoostingClassifier(random_state=0))]:
    oof = np.full(len(yn), np.nan)
    gkf = GroupKFold(n_splits=min(5, len(np.unique(grp))))
    for tr, te in gkf.split(Xn, yn, groups=grp):
        if len(np.unique(yn[tr])) < 2:
            continue
        mm = mk().fit(Xn[tr], yn[tr])
        oof[te] = mm.predict_proba(Xn[te])[:, 1]
    ok = ~np.isnan(oof)
    rows.append({'модель': label, 'n': int(ok.sum()), 'AUC (out-of-fold)': roc_auc_score(yn[ok], oof[ok])})
rows.append({'модель': 'только «в меньшинстве R1»', 'n': len(yn), 'AUC (out-of-fold)': roc_auc_score(yn, F['minority'].to_numpy())})
show(pd.DataFrame(rows).set_index('модель'), 'Предсказание порчи: AUC вне обучающей выборки')
print()
print('Ориентир из литературы: Hao 2026 сообщает AUC ~0.79 для предсказания')
print('вредной конформности по признакам нулевого раунда.')



Предсказание порчи: AUC вне обучающей выборки
------------------------------------------------------------
                              n  AUC (out-of-fold)
модель                                            
логистическая              1208             0.9063
градиентный бустинг        1208             0.9074
только «в меньшинстве R1»  1208             0.7899

Ориентир из литературы: Hao 2026 сообщает AUC ~0.79 для предсказания
вредной конформности по признакам нулевого раунда.


In [9]:
A = load_arms(verbose=False)
rows = []
for arm, g in E.groupby('arm', observed=True):
    if arm not in A:
        continue
    per_case = g.groupby('case').agg(порча=('порча', 'sum'), починка=('починка', 'sum'))
    s = js(A, arm).reindex(per_case.index)
    ok = s.notna()
    per_case, s = (per_case[ok], s[ok])
    hi = s[per_case['порча'] > 0]
    lo = s[per_case['порча'] == 0]
    if len(hi) < 5 or len(lo) < 5:
        continue
    u = stats.mannwhitneyu(hi, lo)
    rows.append({'плечо': arm, 'кейсов с порчей': len(hi), 'без порчи': len(lo), 'Judge с порчей': float(hi.mean()), 'Judge без': float(lo.mean()), 'Δ': float(hi.mean() - lo.mean()), 'p': float(u.pvalue)})
if rows:
    show(pd.DataFrame(rows).set_index('плечо'), 'Judge_score в кейсах с порчей и без')
else:
    print('недостаточно кейсов с порчей для сравнения по плечам')



Judge_score в кейсах с порчей и без
------------------------------------------------------------
                  кейсов с порчей  без порчи  Judge с порчей  Judge без       Δ      p
плечо                                                                                 
P-base                         21         88          0.7811     0.8077 -0.0265 0.2925
P-mixed                        11         76          0.7847     0.7810  0.0037 0.9949
P-strong                       15         86          0.8435     0.8007  0.0428 0.3821
P-weak                         13         97          0.7261     0.7757 -0.0496 0.2912
homog deepseekv4               13         45          0.7830     0.7819  0.0011 0.7583
homog gemini                    6         31          0.7291     0.7138  0.0153 0.5097
homog gpt                      12         38          0.7142     0.7647 -0.0505 0.5778


In [10]:
save(S.reset_index(), 'conformity_and_answer_repair_spoil_repair.csv')
save(res.reset_index(), 'conformity_and_answer_repair_logit.csv')
save(L, 'conformity_and_answer_repair_long.csv')


сохранено: out/conformity_and_answer_repair_spoil_repair.csv  (8 строк)
сохранено: out/conformity_and_answer_repair_logit.csv  (7 строк)
сохранено: out/conformity_and_answer_repair_long.csv  (5415 строк)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\analyses\\panel_mechanisms\\results\\conformity_and_answer_repair_long.csv'